In [31]:
# Load modular data 

import gurobipy as gp
from gurobipy import GRB
import pandas as pd

m = gp.Model("MIP")

from ranDataGen.order_data_loader import load_order_data

# Load from ranDataGen copy; change size and sample to switch datasets
size = "large"
sample = 1

base_dir = f"ranDataGen copy/{size} sized samples/"

itemtypes_path = f"order_itemtypes_{sample}.csv"
quantities_path = f"order_quantities_{sample}.csv"
totes_path = f"orders_totes_{sample}.csv"

# Load baseline data using modular loader
df = load_order_data(
    itemtypes_path=itemtypes_path,
    quantities_path=quantities_path,
    totes_path=totes_path,
    base_dir=base_dir,
)

df

,order,item_slot,itemtype,quantity,tote
0,1,0,5,2,22
1,2,0,2,1,24
2,2,1,3,3,24
3,2,2,4,1,104
4,3,0,5,2,14
...,...,...,...,...,...
512,256,0,1,1,77
513,257,0,5,1,89
514,258,0,1,2,116
515,258,1,6,1,116


# Time Based Model

In [ ]:
# ==================================================
# Faster MIP build: avoid pick[b,item,order,position]
# ==================================================

import gurobipy as gp
from gurobipy import GRB

m = gp.Model("MIP_time_based")

# --- Speed/accuracy knobs (these directly trade accuracy for runtime) ---
m.Params.TimeLimit = 10          # seconds; standard 60
m.Params.MIPGap = 0.20           # 5% optimality gap
m.Params.Presolve = 2
m.Params.Heuristics = 0.9        # 0.2

# ==================================================
# Build inventory items: (item_id, item_type, tote)
# ==================================================

items = []
tote_items = {}

item_id = 0
for _, row in df.iterrows():
    itype = int(row["itemtype"])
    tote = int(row["tote"])
    qty = int(row["quantity"])

    for _ in range(qty):
        item_id += 1
        item = (item_id, itype, tote)
        items.append(item)
        tote_items.setdefault(tote, []).append(item)

N = len(items)
positions = range(1, N + 1)

totes = list(tote_items.keys())
tote_size = {t: len(tote_items[t]) for t in totes}

# ==================================================
# Demand dictionary
# ==================================================

orders = list(df["order"].unique())

demand = {}
for _, row in df.iterrows():
    o = int(row["order"])
    itype = int(row["itemtype"])
    qty = int(row["quantity"])
    demand[(o, itype)] = demand.get((o, itype), 0) + qty

# ==================================================
# Item position on conveyor (assignment model)
# ==================================================

x = m.addVars(
    [(iid, p) for (iid, _, _) in items for p in positions],
    vtype=GRB.BINARY,
    name="x",
)

m.addConstrs(
    (gp.quicksum(x[iid, p] for p in positions) == 1 for (iid, _, _) in items),
    name="one_pos_per_item",
)

m.addConstrs(
    (gp.quicksum(x[iid, p] for (iid, _, _) in items) == 1 for p in positions),
    name="one_item_per_pos",
)

pos_expr = {
    iid: gp.quicksum(p * x[iid, p] for p in positions)
    for (iid, _, _) in items
}

# ==================================================
# Tote contiguity + non-overlap
# ==================================================

start = m.addVars(totes, vtype=GRB.INTEGER, lb=1, ub=N, name="tote_start")

m.addConstrs(
    (pos_expr[iid] >= start[t] for (iid, _, t) in items),
    name="tote_start_lb",
)

m.addConstrs(
    (pos_expr[iid] <= start[t] + tote_size[t] - 1 for (iid, _, t) in items),
    name="tote_start_ub",
)

# Only create ordering binaries for each unordered pair (t1 < t2)
M = N
pairs = [(t1, t2) for idx, t1 in enumerate(totes) for t2 in totes[idx + 1 :]]

z = m.addVars(pairs, vtype=GRB.BINARY, name="tote_order")

# If z[t1,t2]=1 => t1 before t2; else t2 before t1
m.addConstrs(
    (
        start[t1] + tote_size[t1] <= start[t2] + M * (1 - z[t1, t2])
        for (t1, t2) in pairs
    ),
    name="tote_before",
)

m.addConstrs(
    (
        start[t2] + tote_size[t2] <= start[t1] + M * (z[t1, t2])
        for (t1, t2) in pairs
    ),
    name="tote_after",
)

# ==================================================
# Belt assignment (4 belts)
# ==================================================

belts = range(1, 5)

y = m.addVars([(b, o) for b in belts for o in orders], vtype=GRB.BINARY, name="assign")

m.addConstrs((gp.quicksum(y[b, o] for b in belts) == 1 for o in orders), name="assign_one_belt")

# ==================================================
# Picking variables (NO position index)
# pick[b,iid,o] = 1 if belt b uses item iid for order o
# ==================================================

pick = m.addVars(
    [(b, iid, o) for b in belts for (iid, _, _) in items for o in orders],
    vtype=GRB.BINARY,
    name="pick",
)

# Item can be used at most once across all belts/orders
m.addConstrs(
    (
        gp.quicksum(pick[b, iid, o] for b in belts for o in orders) <= 1
        for (iid, _, _) in items
    ),
    name="use_item_once",
)

# Picks only allowed on the belt the order is assigned to
m.addConstrs(
    (pick[b, iid, o] <= y[b, o] for b in belts for (iid, _, _) in items for o in orders),
    name="pick_implies_assign",
)

# Remove invalid picks (order doesn't need that item type)
valid = set(demand.keys())  # (order, itype)
for (iid, itype, _) in items:
    for o in orders:
        if (o, itype) not in valid:
            for b in belts:
                pick[b, iid, o].ub = 0

# Demand satisfaction: for each (order, itemtype), pick exactly qty items of that type
for (o, itype), qty in demand.items():
    m.addConstr(
        gp.quicksum(
            pick[b, iid, o] for b in belts for (iid, it, _) in items if it == itype
        )
        == qty,
        name=f"demand_o{o}_t{itype}",
    )

# ==================================================
# TIME-BASED OBJECTIVE
# TIME >= pick time of every picked item
# ==================================================

TIME = m.addVar(vtype=GRB.CONTINUOUS, lb=0.0, name="completion_time")

# Big-M for time relaxation when pick=0
Mtime = 3 * N + 7.5 * (max(belts) - 1) + 3

m.addConstrs(
    (
        TIME
        >= 3 * pos_expr[iid] + 7.5 * (b - 1) + 3 - Mtime * (1 - pick[b, iid, o])
        for b in belts
        for (iid, _, _) in items
        for o in orders
    ),
    name="time_ge_pick_time",
)

m.setObjective(TIME, GRB.MINIMIZE)

# ==================================================
# Solve
# ==================================================

m.optimize()

# ==================================================
# Print Results + save model output to file
# ==================================================

from pathlib import Path

out_dir = Path("MIP") / "output" / str(size)
out_dir.mkdir(parents=True, exist_ok=True)
output_txt_path = out_dir / f"model_output_{size}_{sample}.txt"

_output_lines = []

def log(s: str = ""):
    print(s)
    _output_lines.append(str(s))

log("\nITEM SEQUENCE ON CONVEYOR")

item_type_names = {
    0: "circle",
    1: "pentagon",
    2: "trapezoid",
    3: "triangle",
    4: "star",
    5: "moon",
    6: "heart",
    7: "cross",
}

sequence = []
for (iid, itype, tote) in items:
    for p in positions:
        if x[iid, p].X > 0.5:
            sequence.append((p, iid, itype, tote))

sequence.sort()
for p, iid, itype, tote in sequence:
    name = item_type_names.get(itype, f"type-{itype}")
    log(f"Position {p}: Item {iid} | Type {itype} ({name}) | Tote {tote}")

log("\nBELT → ORDER ASSIGNMENTS")
for b in belts:
    for o in orders:
        if y[b, o].X > 0.5:
            log(f"Belt {b} processes Order {o}")

log("\nPICKS WITH TIME")

picks_with_time = []
for b in belts:
    for (iid, itype, tote) in items:
        for o in orders:
            if pick[b, iid, o].X > 0.5:
                t_pick = 3 * pos_expr[iid].getValue() + 7.5 * (b - 1) + 3
                picks_with_time.append((t_pick, b, iid, itype, tote, o))

picks_with_time.sort(key=lambda x: x[0])
for t_pick, b, iid, itype, tote, o in picks_with_time:
    log(
        f"Belt {b} picks Item {iid} (Type {itype}, Tote {tote}) for Order {o} → Time = {t_pick:.2f} sec"
    )

if m.SolCount > 0:
    log(f"\nSYSTEM COMPLETION TIME: {TIME.X}")

output_txt_path.write_text("\n".join(_output_lines) + "\n", encoding="utf-8")
log(f"\nModel output saved: {output_txt_path}")


Set parameter TimeLimit to value 10
Set parameter MIPGap to value 0.2
Set parameter Presolve to value 2
Set parameter Heuristics to value 0.9


# CSV Input File

In [24]:
# Shape mapping
shape_map = {
    0: "circle",
    1: "pentagon",
    2: "trapezoid",
    3: "triangle",
    4: "star",
    5: "moon",
    6: "heart",
    7: "cross"
}

belts = range(1,5)

rows = []

for b in belts:
    row = {"conv_num": b}

    # Initialize columns
    for col in shape_map.values():
        row[col] = 0

    # Count picks by item type
    for item_id, item_type, tote in items:

        if item_type not in shape_map:
            continue

        shape_name = shape_map[item_type]

        count = 0

        for o in orders:
            if pick[b, item_id, o].X > 0.5:
                count += 1

        row[shape_name] += count

    rows.append(row)

# Create dataframe
df_picklist = pd.DataFrame(rows)

# Ensure correct column order
df_picklist = df_picklist[
    ["conv_num"] + list(shape_map.values())
]

# Save CSV to size/sample-specific folder
from pathlib import Path

out_dir = Path("MIP") / "output" / str(size)
out_dir.mkdir(parents=True, exist_ok=True)

output_path = out_dir / f"MIP-belt_picklist_{size}_{sample}.csv"
df_picklist.to_csv(output_path, index=False)

print("CSV generated:", output_path)

CSV generated: MIP/output/medium/MIP-belt_picklist_medium_3.csv
